In [ ]:
!pip -q install --no-deps -U bitsandbytes
!pip -q install -U google-genai openai

In [ ]:
# =============================================================================
# STAGE 3b - LLM HARNESS  (zero-shot AND few-shot, NFR SUB-TYPE multi-class)
#
# Built by copying the validated Stage 3 harness and swapping the binary tasks
# for the multi-class NFR sub-type task at three granularity levels (top-4,
# top-6, all-11). It therefore INHERITS every Stage 3 fix: proportional core
# subset, API model discovery, native-Phi loading, verified real prices, the
# two-phase schedule, per-row cost, and the verification gate.
#
# TAXONOMY: Cleland-Huang et al. (2007) PROMISE NFR sub-classes, ALIGNED WITH
# but not identical to ISO/IEC 25010. "look_and_feel", "operational" and
# "legal" are not ISO 25010 characteristics - say "Cleland-Huang (2007),
# aligned with ISO/IEC 25010", never "ISO/IEC 25010 categories".
#
# The FINE-TUNED sub-type comparator is produced by Stage 2 (in_domain_subtype_*
# families). This stage supplies the prompted-LLM side of that comparison.
#
# OUTPUTS -> /kaggle/working/   (this stage's own artefacts - the block further
# down belongs to the retained Stage 3 header and lists Stage 3's files)
#      predictions_subtype.parquet (+ .csv)   subtype_summary.csv
#      subtype_fewshot_effect.csv             subtype_baseline.csv
#      subtype_perclass.csv                   stage3b_failures.csv
#      stage3b_class_distribution.csv         stage3b_manifest.json
#      commercial_availability.json           stage3b_quarantine.csv (if any)
# =============================================================================
# STAGE 3 lineage. This stage was built by copying the validated Stage 3
# binary harness. Stage 3's own header used to be reproduced here in full,
# which meant this file carried a page of claims about the BINARY tasks -
# few-shot at k=2/4, three evaluation frames, Stage 3's file names - that are
# not true of this stage and were read as if they were. Read
# stage3-llm-harness for that history; what is true HERE is stated above.
# =============================================================================

# ############# CELL A ########################################################
# !pip -q install -U google-genai openai
# !pip -q install --no-deps -U bitsandbytes
# #############################################################################

import gc
import glob
import json
import logging
import os
import platform
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
for noisy in ["httpx", "urllib3", "filelock", "huggingface_hub", "google_genai", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)
log = logging.getLogger("stage3")

CONFIG = {
    "mode": "auto",                 # auto = smoke test, then the full run
    "data_dir": None,               # auto-discovered under /kaggle/input
    "out_dir": "/kaggle/working",

    "tasks": ["subtype_top4", "subtype_top6", "subtype_all"],

    # =====================================================================
    # EVALUATION DESIGN  (every value below is justified; none is inherited)
    # =====================================================================
    # CORE SUBSET. A fixed, seed-stable, class-stratified subset of each
    # (task, dataset) group. EVERY model - local and API - is evaluated on
    # exactly these items, so the headline comparison table is like-for-like.
    # (Stage 3's binary tasks use 300; the power argument quoted there is for
    # n=300 and does not carry over to this stage unchanged.)
    # NFR sub-typing has far fewer items than the binary tasks: PROMISE has 524
    # NFRs total, and top-4 covers 353 of them. core_eval_n is capped at
    # 200 so it is a genuine subset rather than "all of it", and so 3 tasks x
    # 200 = 600 fits a single API provider's free daily quota. (stratified()'s
    # min_per_class floor still lifts the all-11 core to 218 items - see the
    # note under EVALUATION SETS.)
    "core_eval_n": 200,

    # FULL SETS. The local models additionally run zero-shot over every item
    # (341 / 415 / 491 for top-4 / top-6 / all-11 after few-shot exemplars are
    # carved out). Reported as a robustness check, and used for the RQ2
    # comparison against the fine-tuned encoders, which were themselves
    # evaluated on the full corpora.
    "run_full_sets": True,

    # FEW-SHOT. The Literature Review promises "zero-shot and few-shot"
    # without fixing k. In this stage k is examples PER CLASS: k=1 already
    # puts 11 exemplars in the prompt for all-11 and k=2 puts 22, and larger k
    # risks overflowing context on the small models - so the ladder here is
    # k in {0, 1, 2} (Stage 3's binary ladder {0, 2, 4, 8} is where
    # saturation is probed). Few-shot runs on the CORE subset, and k=0 is
    # re-scored on the same item set (fewshot_effect / stage3b-repair R2), so
    # the deltas are item-paired and McNemar-testable.
    "fewshot_ks": [1, 2],
    "fewshot_pool_per_class": 3,   # >= max(k)=2, with headroom

    # PROMPT SENSITIVITY sub-study. ONE model here (Qwen2.5-7B),
    # deliberately: the sub-type budget is better spent on few-shot coverage,
    # and the cross-model evidence that prompt-wording effects are not a
    # single-model artefact comes from Stage 3's binary sub-study, which runs
    # two models.
    "run_prompt_substudy": True,
    "substudy_models": ["Qwen/Qwen2.5-7B-Instruct"],
    "substudy_prompts": ["terse", "verbose"],
    # The sub-study runs on EVERY granularity level: conditions_for() iterates
    # all frames. A "substudy_task": "subtype_top4" key used to sit here and
    # implied a restriction, but nothing ever read it - and the committed
    # store carries terse/verbose rows for all three levels. Removed rather
    # than left to mislead the methodology section.

    # =====================================================================
    # MODELS
    # =====================================================================
    "open_models": [
        "Qwen/Qwen2.5-3B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct",
        "microsoft/Phi-4-mini-instruct",
        "HuggingFaceTB/SmolLM3-3B",
        "meta-llama/Llama-3.1-8B-Instruct",
        "google/gemma-2-2b-it",
    ],
    # Tried in order if the primary fails preflight. Phi-4-mini has previously
    # failed to load against transformers 5.x (SlidingWindowCache import); the
    # fallback keeps the Phi family represented rather than silently absent.
    "model_fallbacks": {
        "microsoft/Phi-4-mini-instruct": ["microsoft/Phi-3.5-mini-instruct"],
    },
    "model_kwargs": {
        "google/gemma-2-2b-it": {"attn_implementation": "eager"},
        # Phi ships remote modelling code written against transformers 4.x; it
        # imports SlidingWindowCache, which 5.0 removed. transformers has a
        # native Phi3 implementation, so we use that rather than the Hub copy.
        "microsoft/Phi-4-mini-instruct": {"trust_remote_code": False,
                                          "attn_implementation": "eager"},
        "microsoft/Phi-3.5-mini-instruct": {"trust_remote_code": False,
                                            "attn_implementation": "eager"},
    },

    # QUANTIZATION. Applied uniformly to every local model and recorded on
    # every prediction row.
    #   Why 4-bit at all: Kaggle maps the model to a single T4 (16 GB).
    #     Llama-3.1-8B in fp16 needs ~16 GB and will not fit.
    #   Why uniformly: mixing fp16 for the small models with 4-bit for the large
    #     one would compare models at different numerical precision, which is an
    #     unfair benchmark and impossible to defend.
    #   Cost: NF4 typically costs 1-3 macro-F1 points. That penalty falls on the
    #     LLM side of the comparison, i.e. it makes the thesis claim HARDER to
    #     support, never easier.
    #   Report as: "all open-weight models were quantised to 4-bit NF4 with
    #     double quantisation and fp16 compute, reflecting the free-tier hardware
    #     the cost analysis assumes."
    "quantization": "4bit_nf4",     # "4bit_nf4" | "fp16"
    "require_quantization": True,   # abort rather than silently fall back

    # =====================================================================
    # API TIER
    # =====================================================================
    # ON A RESUMED RUN, READ THIS FIRST.
    # The local models are skipped entirely once their predictions are in the
    # seeded store, but the API tier is NOT complete: its free daily quota cut
    # the committed run short (Gemini answered 60 of 300 items per group), so
    # a re-run with keys present will spend today's quota topping those cells
    # up. That is an improvement - more items, narrower intervals - but it
    # CHANGES the numbers for the API models, so do it before writing them up,
    # not after.
    #   True  = top up the API cells with today's quota (numbers will move)
    #   False = reproduce the committed results exactly, no API calls at all
    "run_hosted": True,

    # PRICE OVERRIDES for models found by live discovery. Discovery can only
    # tell us a model EXISTS, not what it costs, so an unknown model falls back
    # to the cheapest declared rate of its provider - an estimate, not a fact,
    # and RQ3's cost axis would silently inherit it. Look the real published
    # price up and put it here; the run prints a loud warning for every model
    # still using an estimate, and records it in the manifest.
    # Format:  "model-id": (usd_per_1M_input_tokens, usd_per_1M_output_tokens)
    "price_overrides": {
        # Published USD per 1M tokens (input, output), verified August 2026.
        "gemini-3.1-flash-lite":    (0.25, 1.50),
        "gemini-3.1-flash-lite-preview": (0.25, 1.50),
        "gemini-3.5-flash-lite":    (0.30, 2.50),
        "gemini-2.5-flash-lite":    (0.10, 0.40),
        "gemini-3.6-flash":         (1.50, 7.50),
        "gemini-3.5-flash":         (1.50, 9.00),
        "gemini-3.1-pro":           (2.00, 12.00),
        "llama-3.3-70b-versatile":  (0.59, 0.79),
        "llama-3.1-8b-instant":     (0.05, 0.08),
        # OpenRouter ":free" routes bill nothing; 0.0 is the true rate, not a
        # placeholder. Their cost advantage is real but rate-limited.
        "nvidia/nemotron-3-nano-30b-a3b:free": (0.0, 0.0),
        "nvidia/nemotron-nano-9b-v2:free":     (0.0, 0.0),
        "deepseek/deepseek-chat-v3.1:free":    (0.0, 0.0),
        "qwen/qwen3-8b:free":                  (0.0, 0.0),
    },
    # Applied when a discovered model has no override. Output tokens are
    # typically ~4x input; using one number for both understates cost.
    "estimated_output_multiplier": 4.0,

    "hosted_providers": {
        "gemini": {
            "kind": "commercial", "weights": "closed",
            "secret": "GEMINI_API_KEY", "rpm": 10, "daily_cap": 180,
            # Verified against published rates, August 2026. Cheapest first.
            # Gemini 2.0 Flash / 2.0 Flash-Lite were shut down on 1 June 2026 and
            # the entire 1.5 family now returns 404, which is exactly why the
            # first probe found "no free quota" / "model not available" for every
            # named candidate. They are removed rather than left to fail.
            "candidates": [("gemini-2.5-flash-lite", 0.10, 0.40),
                           ("gemini-3.1-flash-lite", 0.25, 1.50),
                           ("gemini-3.5-flash-lite", 0.30, 2.50),
                           ("gemini-3.6-flash", 1.50, 7.50),
                           ("gemini-3.5-flash", 1.50, 9.00),
                           ("gemini-3.1-pro", 2.00, 12.00)],
            # Model names change often and differ by account. If every named
            # candidate fails, the probe asks the API which models this key can
            # actually use and tries those. Guessing names is what made the
            # commercial tier vanish twice.
            "discover": True, "discover_match": ["flash", "gemini"],
        },
        "openai": {
            "kind": "commercial", "weights": "closed",
            "secret": "OPENAI_API_KEY", "rpm": 20, "daily_cap": 900,
            "candidates": [("gpt-4o-mini", 0.15, 0.60), ("gpt-4o", 2.50, 10.00)],
        },
        "groq": {
            "kind": "open_hosted", "weights": "open",
            "secret": "GROQ_API_KEY", "rpm": 25, "daily_cap": 900,
            "candidates": [("llama-3.3-70b-versatile", 0.59, 0.79),
                           ("llama-3.1-8b-instant", 0.05, 0.08)],
            "discover": True, "discover_match": ["llama", "qwen", "gemma"],
        },
        "openrouter": {
            "kind": "open_hosted", "weights": "open",
            "secret": "OPENROUTER_API_KEY", "rpm": 15, "daily_cap": 900,
            "candidates": [("deepseek/deepseek-chat-v3-0324:free", 0.0, 0.0),
                           ("deepseek/deepseek-r1-distill-llama-70b:free", 0.0, 0.0),
                           ("meta-llama/llama-3.3-70b-instruct:free", 0.0, 0.0),
                           ("qwen/qwen-2.5-72b-instruct:free", 0.0, 0.0),
                           ("google/gemma-2-9b-it:free", 0.0, 0.0),
                           ("mistralai/mistral-7b-instruct:free", 0.0, 0.0)],
            "discover": True, "discover_match": [":free"],
        },
    },

    # =====================================================================
    # GENERATION  (deterministic; declare all of this in Chapter 3)
    # =====================================================================
    "temperature": 0.0,             # greedy decoding
    "do_sample": False,             # deterministic: a seed would be redundant
    "max_new_tokens": 12,           # longest valid label is ~4 tokens
    "max_input_chars": 2000,        # longest requirement in either corpus < 2000

    # =====================================================================
    # COSTING & QUALITY GATES
    # =====================================================================
    # AWS g4dn.xlarge (1 x NVIDIA T4 16 GB) - the same accelerator Kaggle
    # provides - ON-DEMAND list price in us-east-1, verified 6 August 2026.
    # The previous value of 0.35 matched no published rate and understated
    # local inference cost by roughly a third.
    #   on-demand $0.5260/hr   <- used here (defensible list price)
    #   spot      $0.3162/hr   <- reported as a sensitivity in Stage 5
    # This is an IMPUTED rental cost for the hardware, not a market price for
    # the model, and it assumes sequential unbatched inference. Both caveats
    # belong in RQ3's threats to validity.
    "open_gpu_usd_per_hour": 0.5260,
    "open_gpu_usd_per_hour_spot": 0.3162,
    "gpu_price_source": ("AWS EC2 g4dn.xlarge (1x T4), on-demand, us-east-1, "
                         "verified 2026-08-06; spot $0.3162/hr same date"),

    # Provenance for every rate above, so Chapter 3 can cite rather than assert.
    "pricing_verified_on": "2026-08-06",
    "pricing_notes": {
        "gemini-3.1-flash-lite": ("Google standard tier $0.25/$1.50 per 1M "
                                  "tokens. One tracker reports a GA rate of "
                                  "$0.125/$0.75; the higher published standard "
                                  "rate is used here as the conservative choice."),
        "groq-llama-3.3-70b-versatile": "Groq published rate $0.59/$0.79 per 1M.",
        "openrouter-free-routes": "':free' routes bill $0.00 but are rate limited.",
        "local_models": ("Imputed AWS T4 rental; not a market price. Batched "
                         "serving would reduce this by roughly an order of "
                         "magnitude."),
    },

    # A model answering fewer than half its prompts in parseable form has not
    # scored badly - it has not been measured. Such cells are quarantined with a
    # stated reason instead of entering the results table as macro-F1 = 0.000.
    "min_parse_rate": 0.50,

    "n_boot": 1000,                 # bootstrap replicates for the 95% CIs
    "flush_every": 50,              # checkpoint interval (rows)
    "smoke_n_per_group": 12,
}

# Byte-identical to Stage 2's canonical 28-column list, train_epochs included.
# The list here used to omit train_epochs while both stages' comments claimed
# the schema was shared verbatim; it is 0 for a prompted model (nothing is
# trained), and load_store() back-fills it on a store written before the
# column existed.
STORE_COLS = ["model_tag", "model_type", "approach", "class_weighting",
              "task", "dataset", "eval_regime", "family", "fold",
              "split", "prompt_id", "shot_k",
              "id", "y_true", "y_pred", "parse_ok",
              "prompt_tokens", "completion_tokens", "latency_s",
              "train_time_s", "train_epochs", "price_in_per_mtok",
              "price_out_per_mtok",
              "cost_usd", "cost_basis", "quantization", "timestamp", "raw_output"]

# The exact 12-column view requested for the thesis appendix. It is a PROJECTION
# of the canonical store, not a replacement: Stages 2/4/5 need task, fold,
# prompt_id, shot_k, parse_ok and model_type, none of which appear here. Emitting
# both means the appendix gets the tidy schema while the pipeline keeps the
# columns it cannot run without.
UNIFIED_VIEW_COLS = ["model_name", "dataset", "split", "input_id", "true_label",
                     "predicted_label", "latency_s", "prompt_tokens",
                     "completion_tokens", "total_cost_usd", "timestamp",
                     "eval_regime"]
VIEW_RENAME = {"model_tag": "model_name", "id": "input_id", "y_true": "true_label",
               "y_pred": "predicted_label", "cost_usd": "total_cost_usd"}

# Cleland-Huang et al. (2007) PROMISE NFR sub-classes.
CATEGORIES_ALL = ["availability", "fault_tolerance", "legal", "look_and_feel",
                  "maintainability", "operational", "performance", "portability",
                  "scalability", "security", "usability"]
TOP4 = ["security", "usability", "operational", "performance"]
TOP6 = TOP4 + ["look_and_feel", "availability"]
LABELSETS = {"subtype_all": CATEGORIES_ALL, "subtype_top6": TOP6, "subtype_top4": TOP4}

# Which fine-tuned fold each LLM evaluation cell is comparable to, so RQ2 joins
# are explicit instead of being reconstructed from fold-name substrings.
COMPARABLE_FT_FOLD = {
    # Each sub-type task compares against Stage 2's pooled in-domain CV for the
    # same granularity level. Stage 4 groups on task+eval_regime, so a stable
    # marker is enough here.
    ("subtype_top4", "promise"): "indomain_subtype_top4_pooled",
    ("subtype_top6", "promise"): "indomain_subtype_top6_pooled",
    ("subtype_all", "promise"): "indomain_subtype_all_pooled",
}

OUT = CONFIG["out_dir"]
Path(OUT).mkdir(parents=True, exist_ok=True)
HOSTED_PRICES = {}
# Models whose token price is an ESTIMATE rather than a published rate. Any
# model left in here at the end makes the RQ3 cost figure for that model
# indicative only, and the run says so loudly.
ESTIMATED_PRICES = set()
FAILURES = []


# =============================================================================
# io
# =============================================================================
def find_data_dir(explicit=None):
    if explicit and (Path(explicit) / "unified.parquet").exists():
        return str(explicit)
    hits = sorted(glob.glob("/kaggle/input/**/unified.parquet", recursive=True))
    hits += sorted(glob.glob("/kaggle/input/**/unified.csv", recursive=True))
    if not hits:
        raise FileNotFoundError("unified.parquet not found under /kaggle/input. "
                                "Add Stage 1's committed output as an input.")
    return str(Path(hits[0]).parent)


def read_table(base):
    """keep_default_na=False: a stored "n/a"-like string must not become NaN on
    the resume path, or completed work is silently lost."""
    for ext in (".parquet", ".csv"):
        if os.path.exists(base + ext):
            try:
                if ext == ".parquet":
                    return pd.read_parquet(base + ext)
                return pd.read_csv(base + ext, keep_default_na=False, na_values=[""])
            except Exception:
                pass
    return None


def write_table(df, base):
    df.to_csv(base + ".csv", index=False)
    try:
        df.to_parquet(base + ".parquet", index=False)
    except Exception as e:
        log.warning("parquet write skipped (%s)", e)


STORE_PATH = os.path.join(OUT, "predictions_subtype")


# =============================================================================
# environment + credentials
# =============================================================================
def check_environment():
    print("=" * 70); print("ENVIRONMENT"); print("=" * 70)
    print(f"  torch          : {torch.__version__}")
    print(f"  CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU            : {torch.cuda.get_device_name(0)}")
    else:
        print("  !! NO GPU -> Settings -> Accelerator -> GPU T4 x2, then restart.")
    import transformers
    print(f"  transformers   : {transformers.__version__}")
    major = int(transformers.__version__.split(".")[0])
    dtype_kw = {"dtype": torch.float16} if major >= 5 else {"torch_dtype": torch.float16}
    try:
        from transformers.utils import is_bitsandbytes_available
        use_4bit = bool(is_bitsandbytes_available())
    except Exception:
        use_4bit = False
    print(f"  4-bit (bnb)    : {use_4bit}" + ("" if use_4bit else "   -> fp16 fallback"))
    print("=" * 70)
    return dtype_kw, use_4bit


DTYPE_KW, USE_4BIT = check_environment()


def get_secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return os.environ.get(label)


HF_TOKEN = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    log.info("HF_TOKEN loaded - gated models can be checked.")

HOSTED = {}
PROBE_LOG = []        # every probe attempt, for commercial_availability.json
if CONFIG["run_hosted"]:
    for _name, _spec in CONFIG["hosted_providers"].items():
        _key = get_secret(_spec["secret"])
        if _key:
            HOSTED[_name] = {"key": _key, **_spec}
            log.info("%s key loaded (%s / %s weights).",
                     _name, _spec["kind"], _spec["weights"])
        else:
            PROBE_LOG.append({"provider": _name, "kind": _spec["kind"],
                              "model": None, "result": "no API key in Kaggle Secrets"})
            log.info("No %s secret -> %s skipped.", _spec["secret"], _name)


# =============================================================================
# data + evaluation frames
# =============================================================================
CONFIG["data_dir"] = find_data_dir(CONFIG["data_dir"])
uni = read_table(str(Path(CONFIG["data_dir"]) / "unified"))
log.info("Loaded unified corpus: %d rows from %s", len(uni), CONFIG["data_dir"])

# Table 1 for the sub-type stage: class distribution across the three
# granularity levels. This is the analogue of Stage 3's dataset table.
_nfr = uni[uni["label_nfr_subtype"].notna()].copy()
_dist = _nfr["label_nfr_subtype"].astype(str).value_counts()
_t1 = pd.DataFrame({"n": _dist})
_t1["pct_of_nfr"] = (100 * _t1["n"] / _t1["n"].sum()).round(1)
_t1["in_top4"] = _t1.index.isin(TOP4)
_t1["in_top6"] = _t1.index.isin(TOP6)
_t1.to_csv(os.path.join(OUT, "stage3b_class_distribution.csv"))
print("\nTable 1 - NFR sub-type class distribution (PROMISE):")
print(_t1.to_string())
print(f"  total NFRs: {int(_t1['n'].sum())}  |  classes: {len(_t1)}")
print("  The tail is heavy: 'portability' and 'legal' have ~12-15 examples,")
print("  which is why the all-11 task is hard for every model. Report the")
print("  per-class support alongside macro-F1 so the committee sees the cause.")


def build_frames(uni):
    """One evaluation frame per granularity level, all drawn from the PROMISE
    NFR pool. SecReq has no sub-type annotation, so it does not appear here."""
    df = uni.copy()
    df["source_dataset"] = df["source_dataset"].astype(str).str.lower().str.strip()
    if "label_nfr_subtype" not in df.columns:
        raise KeyError("unified corpus has no label_nfr_subtype column. Re-run "
                       "Stage 1 (the current Stage 1 builds sub-type labels).")
    nfr = df[df["label_nfr_subtype"].notna()].copy()
    nfr["y_true"] = nfr["label_nfr_subtype"].astype(str)

    out = {}
    for task in CONFIG["tasks"]:
        ls = LABELSETS[task]
        sub = nfr[nfr["y_true"].isin(ls)][["id", "text", "y_true"]].copy()
        sub = sub.dropna(subset=["text"]).reset_index(drop=True)
        out[(task, "promise")] = sub
    return out


def carve_pool(frames, k):
    """Few-shot exemplars are removed from the evaluation set. Without this the
    same requirement could appear as both an exemplar and a test item."""
    pools, clean = {}, {}
    for key, fr in frames.items():
        picks = [g.sample(min(k, len(g)), random_state=SEED)
                 for _, g in fr.groupby("y_true")]
        pool = pd.concat(picks) if picks else fr.iloc[0:0]
        pools[key] = list(zip(pool["text"].tolist(), pool["y_true"].tolist()))
        clean[key] = fr[~fr["id"].isin(set(pool["id"]))].reset_index(drop=True)
    return clean, pools


def stratified(df, n, min_per_class=10):
    """Draw n items PRESERVING the corpus class proportions.

    This must be proportional, not balanced. An equal-sized draw per class
    would rewrite the class prior the tasks are defined over: a balanced draw
    across eleven sub-types would make 'portability' (2.3% of NFRs) as common
    as 'security' (23.9%), and macro-F1 on such a subset would be incomparable
    with macro-F1 on the full frame and with the fine-tuned baselines, which
    were evaluated on the full corpora.

    A floor per class (min_per_class, capped at the class's own support)
    keeps macro-F1 defined when a class is very rare. The floor is applied
    AFTER the proportional allocation and WITHOUT renormalisation, so the
    returned sample can exceed n and over-represent ultra-rare classes
    relative to the frame - negligible for the binary tasks; on the sub-types
    it is why the all-11 core holds 218 items rather than 200. The EVALUATION
    SETS block printed at start-up reports the realised sizes and minority
    shares, and cross-tier comparisons are made on explicitly stated item
    sets (like-for-like tables), never on the raw core.
    """
    if n is None or len(df) <= n:
        return df.reset_index(drop=True)

    counts = df["y_true"].value_counts()
    # proportional allocation, largest-remainder rounding so the parts sum to n
    exact = counts / counts.sum() * n
    alloc = np.floor(exact).astype(int)
    remainder = n - int(alloc.sum())
    for cls in (exact - alloc).sort_values(ascending=False).index[:remainder]:
        alloc[cls] += 1

    # floor: never let a class vanish entirely, or macro-F1 is undefined for it
    for cls in alloc.index:
        alloc[cls] = min(max(int(alloc[cls]), min_per_class), int(counts[cls]))

    parts = [g.sample(int(alloc[cls]), random_state=SEED)
             for cls, g in df.groupby("y_true") if alloc.get(cls, 0) > 0]
    out = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return out


_raw = build_frames(uni)
_raw = {k: v for k, v in _raw.items() if len(v) > 0}
if not _raw:
    raise RuntimeError("All evaluation frames are empty; check the corpus schema.")
FRAMES, POOL = carve_pool(_raw, CONFIG["fewshot_pool_per_class"])

# THE CORE SUBSET. Fixed, seed-stable, class-stratified. Every model - local
# and API - is evaluated on these items (API tiers as far as their quota
# reaches), so cross-tier comparison is possible. Few-shot also runs here,
# which makes k=0/1/2 item-paired and therefore McNemar-testable.
CORE = {k: stratified(v, CONFIG["core_eval_n"]) for k, v in FRAMES.items()}
CORE_IDS = {k: set(v["id"]) for k, v in CORE.items()}

print("\n" + "=" * 70); print("EVALUATION SETS"); print("=" * 70)
print(f"  {'task/dataset':21s} {'full n':>7s} {'core n':>7s}   "
      f"{'minority % full':>15s} {'minority % core':>15s}")
for (t, d), fr in FRAMES.items():
    core = CORE[(t, d)]
    minority = fr["y_true"].value_counts().idxmin()
    pf = 100 * (fr["y_true"] == minority).mean()
    pc = 100 * (core["y_true"] == minority).mean()
    flag = "" if abs(pf - pc) < 3.0 else "   <-- PRIOR DISTORTED"
    print(f"  {t + '/' + d:21s} {len(fr):7d} {len(core):7d}   "
          f"{pf:14.1f}% {pc:14.1f}%{flag}")
    print(f"      full={dict(fr['y_true'].value_counts())}  "
          f"core={dict(core['y_true'].value_counts())}")
print("\n  The core subset is drawn PROPORTIONALLY, so it carries the corpus")
print("  class prior. One stated deviation: stratified()'s min_per_class floor")
print("  lifts the rarest classes to the whole of their (tiny) support, which")
print("  is why the all-11 core is 218 items rather than 200 and portability")
print("  sits at ~4.1% of CORE against ~1.8% of the frame. stage3b-repair")
print("  reports FULL-scope numbers as the headline for local models and uses")
print("  CORE only for cross-tier comparability, which bounds the effect.")
print("=" * 70)


# =============================================================================
# prompts + parsing
# =============================================================================
def _label_menu(labelset):
    return ", ".join(f'"{c}"' for c in labelset)


def prompt_instruction(prompt_id, task):
    ls = LABELSETS[task]
    human = ", ".join(c.replace("_", " ") for c in ls)
    if prompt_id == "terse":
        return (f"Which quality attribute does this non-functional requirement "
                f"describe? Reply with exactly one of: {_label_menu(ls)}.")
    if prompt_id == "verbose":
        return ("Task: a software NON-FUNCTIONAL requirement describes a quality "
                "attribute or constraint rather than a behaviour. Decide which "
                "single quality attribute it primarily describes. The available "
                f"categories are: {human}. Consider the dominant concern of the "
                "sentence, not incidental wording. Respond with exactly one of "
                f"these labels: {_label_menu(ls)}.")
    return ("You are classifying a NON-FUNCTIONAL software requirement into one "
            f"quality sub-type.\nChoose exactly one of: {_label_menu(ls)}.\n"
            "Answer with only that label and nothing else.")


def pick_examples(pool, k, labelset):
    """Multi-class: k exemplars PER CLASS, class-balanced and deterministic."""
    if k <= 0 or not pool:
        return []
    by_cls = {}
    for txt, y in pool:
        if y in labelset:
            by_cls.setdefault(y, []).append((txt, y))
    classes = sorted(by_cls)
    target = k * len(classes)
    out, i = [], 0
    while len(out) < target and any(by_cls.values()) and i < 10 * target:
        c = classes[i % len(classes)]
        if by_cls[c]:
            out.append(by_cls[c].pop(0))
        i += 1
    return out


def build_prompt(task, text, prompt_id, examples):
    instr = prompt_instruction(prompt_id, task)
    text = str(text)[: CONFIG["max_input_chars"]]
    shots = ""
    if examples:
        blocks = [f'Requirement:\n"""{str(t)[:CONFIG["max_input_chars"]]}"""\nAnswer: {y}'
                  for t, y in examples]
        shots = "Examples:\n\n" + "\n\n".join(blocks) + "\n\nNow classify this one:\n"
    return f'{instr}\n\n{shots}Requirement:\n"""{text}"""\n\nAnswer:'


_THINK = re.compile(r"<think>.*?</think>", re.S)


def clean_output(raw):
    s = str(raw)
    if "<think>" in s and "</think>" not in s:
        return ""                      # truncated mid-thought: no answer given
    return _THINK.sub(" ", s).replace("<think>", " ").replace("</think>", " ").strip()


def parse_label(task, raw):
    """Multi-class parse: the label whose surface form appears EARLIEST wins.

    BYTE-FOR-BYTE the rule stage3b-repair's parse_label_positional applies, so
    the harness's own tables and the repaired store cannot disagree. The
    previous longest-first scan was position-blind: for "security, not
    maintainability" it returned maintainability, because the longer label is
    tested first. Earliest-position returns the label the model led with. Ties
    at the same offset go to the LONGER surface form, which preserves the
    original protection against 'fault_tolerance' being shadowed by a shorter
    substring. Both 'look_and_feel' and 'look and feel' forms are accepted.
    Returns (label_or_None, parsed_ok)."""
    if raw is None:
        return None, False
    r = clean_output(raw).lower()
    if not r:
        return None, False
    best = None
    for c in LABELSETS[task]:
        for surface in (c, c.replace("_", " ")):
            i = r.find(surface)
            if i >= 0 and (best is None or i < best[0]
                           or (i == best[0] and len(surface) > best[1])):
                best = (i, len(surface), c)
    return (best[2], True) if best else (None, False)


# =============================================================================
# store
# =============================================================================
def seed_store_from_inputs():
    """Copy the committed prediction store from /kaggle/input into out_dir.

    /kaggle/working is EMPTY at the start of every Kaggle session, while the
    previous run's store lives in that notebook's committed output (mounted
    read-only under /kaggle/input). Without this step the resume logic finds
    nothing and re-runs EVERY prediction - hours of GPU inference - even when
    only scoring or verification code changed. With it, completed items are
    skipped and only missing ones (e.g. API items lost to rate limits) are
    attempted again.

    Only runs when out_dir has no store yet, so it can never overwrite work in
    progress. The glob matches predictions_subtype.* exactly: the repaired
    store (predictions_subtype_repaired) and the appendix view
    (predictions_subtype_harness) are different artefacts and deliberately not
    eligible.
    """
    if any(os.path.exists(STORE_PATH + ext) for ext in (".parquet", ".csv")):
        return False
    hits = sorted(glob.glob("/kaggle/input/**/predictions_subtype.parquet", recursive=True))
    hits += sorted(glob.glob("/kaggle/input/**/predictions_subtype.csv", recursive=True))
    if not hits:
        log.info("No previous prediction store under /kaggle/input - starting fresh.")
        return False
    df = read_table(str(Path(hits[0]).with_suffix("")))
    if df is None or "model_tag" not in df.columns:
        log.warning("Found %s but could not read it; starting fresh.", hits[0])
        return False
    write_table(df, STORE_PATH)
    log.info("SEEDED %d rows from the committed store (%s). Completed items "
             "will be skipped by the resume logic.", len(df), hits[0])
    return True


def load_store():
    df = read_table(STORE_PATH)
    if df is None:
        return pd.DataFrame(columns=STORE_COLS)
    # Conform an older store to the canonical 28-column schema: train_epochs
    # joined the shared list after the first committed run and is 0 by
    # construction for a prompted model.
    if "train_epochs" not in df.columns:
        df["train_epochs"] = 0
    df = df[STORE_COLS]
    log.info("Resuming: %d predictions already stored.", len(df))
    return df


def done_keys(store):
    if len(store) == 0:
        return set()
    return set(zip(store.model_tag, store.prompt_id, store.shot_k,
                   store.task, store.dataset, store.id))


def save_store(rows, store):
    if rows:
        new = pd.DataFrame(rows, columns=STORE_COLS)
        # Concatenating onto an empty frame triggers a pandas FutureWarning about
        # all-NA dtype inference. Skipping the concat when the store is empty
        # avoids it without suppressing warnings globally.
        store = new if len(store) == 0 else pd.concat([store, new], ignore_index=True)
        rows.clear()
    write_table(store, STORE_PATH)
    return store


def row_cost_usd(basis, n_in, n_out, dt, price):
    """Exact cost of ONE prediction, in USD.

    api_tokens  : billed input/output tokens at the provider's published rate.
    gpu_seconds : amortised rental of the accelerator actually used, i.e.
                  wall-clock seconds x hourly rate. This is an imputed cost, not
                  a market price, and it assumes sequential unbatched inference;
                  both caveats belong in the RQ3 threats to validity.
    """
    if basis == "api_tokens":
        return (n_in / 1e6) * float(price[0]) + (n_out / 1e6) * float(price[1])
    return (float(dt) / 3600.0) * CONFIG["open_gpu_usd_per_hour"]


def make_row(tag, tier, cond, task, ds, item, y_pred, ok, n_in, n_out, dt, raw,
             basis, price=(0.0, 0.0), quant=None):
    return [tag, tier, "prompted", "not_applicable",
            task, ds, "prompted", "llm_eval",
            COMPARABLE_FT_FOLD.get((task, ds), "unmapped"),
            cond["split"], cond["prompt_id"], cond["shot_k"],
            item.id, item.y_true, y_pred if y_pred is not None else "UNPARSED",
            bool(ok), int(n_in), int(n_out), round(float(dt), 5),
            0.0, 0, float(price[0]), float(price[1]),
            round(row_cost_usd(basis, n_in, n_out, dt, price), 10), basis,
            quant or (CONFIG["quantization"] if basis == "gpu_seconds" else "api_fp_unknown"),
            time.strftime("%Y-%m-%dT%H:%M:%S"), str(raw)[:120]]


# =============================================================================
# conditions
# =============================================================================
# Phase 1 = everything the paper cannot be written without. Phase 2 = the
# robustness extras. The runner completes phase 1 for EVERY model before any
# model starts phase 2, so a session that is cut short still leaves a complete,
# like-for-like results table rather than three finished models and three empty
# ones. The cost is one extra model load per model (~40 s), which is cheap
# insurance against losing a four-hour run.
PHASE_PRIMARY, PHASE_EXTRA = 1, 2


def conditions_for(model_id, is_hosted=False):
    """Experimental conditions for one model, tagged with their phase."""
    conds = [
        # --- phase 1: the core, comparable across every model ---------------
        {"split": "zero_shot", "prompt_id": "base", "shot_k": 0,
         "scope": "core", "phase": PHASE_PRIMARY},
    ]
    for k in CONFIG["fewshot_ks"]:
        conds.append({"split": "few_shot", "prompt_id": "base", "shot_k": k,
                      "scope": "core", "phase": PHASE_PRIMARY})

    if is_hosted:
        # API tier stops here: the free daily quota is spent entirely on the
        # comparable zero-shot condition. Note the realised plan is 618
        # requests, not 3 x core_eval_n = 600: stratified()'s min_per_class
        # floor lifts the all-11 core to 218 items.
        return [c for c in conds if c["shot_k"] == 0]

    # --- phase 2: local-only extras ----------------------------------------
    if CONFIG["run_full_sets"]:
        conds.append({"split": "zero_shot", "prompt_id": "base", "shot_k": 0,
                      "scope": "full", "phase": PHASE_EXTRA})
    if CONFIG["run_prompt_substudy"] and model_id in CONFIG["substudy_models"]:
        for pid in CONFIG["substudy_prompts"]:
            conds.append({"split": "zero_shot", "prompt_id": pid, "shot_k": 0,
                          "scope": "core", "phase": PHASE_EXTRA})
    return conds


def build_todo(tag, conditions, done, hard_cap=None, phase=None):
    todo, queued = [], set()
    for cond in conditions:
        if phase is not None and cond.get("phase", PHASE_PRIMARY) != phase:
            continue
        for (task, ds), fr in FRAMES.items():
            sub = CORE[(task, ds)] if cond["scope"] == "core" else fr
            if hard_cap is not None:
                sub = stratified(sub, hard_cap)
            exs = pick_examples(POOL.get((task, ds), []), cond["shot_k"], LABELSETS[task])
            for r in sub.itertuples(index=False):
                key = (tag, cond["prompt_id"], cond["shot_k"], task, ds, r.id)
                # `queued` dedupes within THIS call. The smoke test runs every
                # condition in one batch, where the core and full zero-shot
                # conditions overlap and `done` (the persisted store) cannot
                # see items queued a moment ago - without this, each
                # overlapping item is generated twice.
                if key not in done and key not in queued:
                    queued.add(key)
                    todo.append((cond, task, ds, exs, r))
    return todo


# =============================================================================
# local open-weight models
# =============================================================================
def load_open_model(model_id):
    """Load one open-weight model at the precision declared in CONFIG.

    Precision is UNIFORM across models by design. Mixing fp16 for the small
    models with 4-bit for the large one would benchmark models at different
    numerical precision, which cannot be defended in a comparison study."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    over = CONFIG.get("model_kwargs", {}).get(model_id, {})
    trc = over.get("trust_remote_code", True)
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trc)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(device_map={"": 0}, trust_remote_code=trc, **DTYPE_KW)
    kw.update({k: v for k, v in over.items() if k != "trust_remote_code"})

    if CONFIG["quantization"] == "4bit_nf4":
        if not USE_4BIT:
            if CONFIG["require_quantization"]:
                raise RuntimeError(
                    "CONFIG requests 4-bit NF4 but bitsandbytes is unavailable. "
                    "Silently falling back to fp16 would (a) exhaust the 16 GB T4 "
                    "on the 8B model and (b) benchmark models at mixed precision. "
                    "Install it in CELL A, or set quantization='fp16' deliberately.")
            log.warning("bitsandbytes unavailable; running fp16 (declared).")
        else:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True)

    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.eval()
    return tok, model


def render_chat(tok, prompt):
    """Render to TEXT and switch reasoning OFF - SmolLM3/Qwen3-style models
    otherwise spend the whole token budget inside <think> and never emit a
    label."""
    variants = [
        ([{"role": "system", "content": "/no_think"}, {"role": "user", "content": prompt}],
         {"enable_thinking": False}),
        ([{"role": "system", "content": "/no_think"}, {"role": "user", "content": prompt}], {}),
        ([{"role": "user", "content": prompt}], {"enable_thinking": False}),
        ([{"role": "user", "content": prompt}], {}),
    ]
    for msgs, extra in variants:
        try:
            return tok.apply_chat_template(msgs, add_generation_prompt=True,
                                           tokenize=False, **extra)
        except Exception:
            continue
    return prompt


def open_generate(tok, model, prompt):
    """Real latency (CUDA-synchronised) and real token counts. No placeholders."""
    try:
        enc = tok(render_chat(tok, prompt), return_tensors="pt", add_special_tokens=False)
    except Exception:
        enc = tok(prompt, return_tensors="pt")
    ids = enc["input_ids"].to(model.device)
    attn = enc.get("attention_mask")
    attn = attn.to(model.device) if attn is not None else None
    n_in = int(ids.shape[-1])
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=attn,
                             max_new_tokens=CONFIG["max_new_tokens"],
                             do_sample=False, pad_token_id=tok.pad_token_id)
    if DEVICE == "cuda":
        torch.cuda.synchronize()       # without this the timing is meaningless
    dt = time.time() - t0
    gen = out[0, n_in:]
    return tok.decode(gen, skip_special_tokens=True).strip(), n_in, int(gen.shape[-1]), dt


def preflight_models(model_ids):
    """Check licence, download AND a real 1-token generation before the long
    run. The previous preflight only read the config, so Phi passed here and
    then crashed two hours in."""
    ok, dropped = [], []
    print("\n" + "=" * 70); print("MODEL PREFLIGHT (config + real generation)"); print("=" * 70)
    for mid in model_ids:
        # Try the primary model, then any declared fallback, so a model FAMILY
        # named in the Literature Review is never silently absent from results.
        for cand in [mid] + CONFIG.get("model_fallbacks", {}).get(mid, []):
            try:
                tok, model = load_open_model(cand)
                _ = open_generate(tok, model, "Reply with one word: security")
                ok.append(cand)
                print(f"  OK    {cand}" + ("" if cand == mid else f"   (fallback for {mid})"))
                if cand != mid:
                    dropped.append({"model": mid, "reason": "preflight failed",
                                    "replaced_by": cand})
                del tok, model
                gc.collect(); torch.cuda.empty_cache()
                break
            except Exception as e:
                msg = str(e)
                why = ("licence not accepted / gated" if "gated" in msg.lower()
                       or "401" in msg or "403" in msg else
                       "not found" if "404" in msg else msg.split("\n")[0][:70])
                print(f"  SKIP  {cand}\n          reason: {why}")
                gc.collect(); torch.cuda.empty_cache()
        else:
            dropped.append({"model": mid, "reason": why, "replaced_by": None})
    print(f"\n  {len(ok)} of {len(model_ids)} model(s) will run.")
    if dropped:
        print("  Dropped models are recorded in stage3_manifest.json and must be")
        print("  reported in the thesis rather than silently omitted.")
    print("=" * 70)
    return ok, dropped


def run_open_model(model_id, store, rows, phase=None):
    tag = model_id.split("/")[-1].lower()
    todo = build_todo(tag, conditions_for(model_id), done_keys(store), phase=phase)
    if not todo:
        log.info("[%s] phase %s already complete - skipping.", tag, phase)
        return store
    log.info("[%s] loading (%s) ... %d predictions queued for phase %s",
             tag, CONFIG["quantization"], len(todo), phase)
    tok, model = load_open_model(model_id)
    t0, since = time.time(), 0
    for i, (cond, task, ds, exs, r) in enumerate(todo, 1):
        try:
            raw, n_in, n_out, dt = open_generate(
                tok, model, build_prompt(task, r.text, cond["prompt_id"], exs))
            y_pred, ok = parse_label(task, raw)
            rows.append(make_row(tag, "open_local", cond, task, ds, r,
                                 y_pred, ok, n_in, n_out, dt, raw, "gpu_seconds"))
        except Exception as e:
            # A hard failure is recorded separately and NOT written into the
            # prediction store as a zero row.
            FAILURES.append({"model_tag": tag, "task": task, "dataset": ds,
                             "id": r.id, "prompt_id": cond["prompt_id"],
                             "shot_k": cond["shot_k"], "error": str(e)[:200]})
        since += 1
        if since >= CONFIG["flush_every"]:
            store = save_store(rows, store); since = 0
        if i % 250 == 0:
            rate = i / max(1e-9, time.time() - t0)
            log.info("[%s] %d/%d (%.1f/s, ~%.0f min left)",
                     tag, i, len(todo), rate, (len(todo) - i) / max(rate, 1e-9) / 60)
    store = save_store(rows, store)
    log.info("[%s] done: %d predictions in %.1f min.", tag, len(todo), (time.time() - t0) / 60)
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    return store


# =============================================================================
# API tier
# =============================================================================
def hosted_tag(provider, model_id):
    short = model_id.split("/")[-1].replace(":free", "")
    return short if provider in ("gemini", "openai") else f"{provider}-{short}"


def make_client(provider, key):
    if provider == "gemini":
        from google import genai
        return genai.Client(api_key=key)
    from openai import OpenAI
    base = {"groq": "https://api.groq.com/openai/v1",
            "openrouter": "https://openrouter.ai/api/v1",
            "openai": "https://api.openai.com/v1"}[provider]
    return OpenAI(api_key=key, base_url=base)


def hosted_call_once(provider, client, model_id, prompt):
    """Returns (text, tok_in, tok_out, latency_s). Token counts come from the
    provider's usage object, so they are real billed counts."""
    if provider == "gemini":
        from google.genai import types
        kw = dict(temperature=0.0, max_output_tokens=CONFIG["max_new_tokens"])
        if model_id.startswith("gemini-2.5"):
            try:
                kw["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
            except Exception:
                pass
        t0 = time.time()
        r = client.models.generate_content(model=model_id, contents=prompt,
                                           config=types.GenerateContentConfig(**kw))
        dt = time.time() - t0
        um = getattr(r, "usage_metadata", None)
        return ((r.text or "").strip(),
                int(getattr(um, "prompt_token_count", 0) or 0),
                int(getattr(um, "candidates_token_count", 0) or 0), dt)
    t0 = time.time()
    r = client.chat.completions.create(
        model=model_id, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, max_tokens=CONFIG["max_new_tokens"])
    dt = time.time() - t0
    u = getattr(r, "usage", None)
    return ((r.choices[0].message.content or "").strip(),
            int(getattr(u, "prompt_tokens", 0) or 0),
            int(getattr(u, "completion_tokens", 0) or 0), dt)


def classify_hosted_error(msg):
    m = msg.lower()
    if "limit: 0" in m or "no_free_quota" in m:
        return "no free quota on this account"
    if "429" in m or "rate" in m or "resource_exhausted" in m or "quota" in m:
        return "rate limited"
    if "404" in m or "not found" in m or "does not exist" in m or "decommission" in m:
        return "model not available"
    if "failed_precondition" in m or "location" in m or "region" in m:
        return "not available in this region"
    if "401" in m or "403" in m or "invalid" in m or "authentication" in m:
        return "invalid or unauthorised key"
    if "no module named" in m:
        return "client package missing (add it to CELL A)"
    return msg[:70]


def discover_models(provider, client, match_terms):
    """Ask the provider which models THIS key can actually use.

    Hard-coded model names go stale constantly and differ between accounts:
    Gemini reported 'no free quota' / 'model not available' for four named
    candidates and OpenRouter for two, which silently removed an entire tier
    from the benchmark twice. Discovery turns that permanent failure into a
    recoverable one. Returns a list of model ids, best-guess first.
    """
    ids = []
    try:
        if provider == "gemini":
            for m in client.models.list():
                name = getattr(m, "name", "") or ""
                actions = getattr(m, "supported_actions", None) or []
                if actions and "generateContent" not in actions:
                    continue
                ids.append(name.replace("models/", ""))
        else:
            for m in client.models.list().data:
                ids.append(getattr(m, "id", ""))
    except Exception as e:
        log.info("%s: model discovery unavailable (%s)", provider, str(e)[:80])
        return []

    keep = [i for i in ids if i and any(t.lower() in i.lower() for t in match_terms)]

    # Rank cheaper/smaller tiers first. Matching must be on TOKEN boundaries,
    # not raw substrings: "mini" is a substring of "ge-mini", so a naive
    # `"mini" in name` marks every Gemini model as a small tier and the ranking
    # collapses. Split on non-alphanumerics and compare whole tokens.
    small = {"flash", "mini", "instant", "lite", "small", "nano"}

    def rank(model_id):
        tokens = set(re.split(r"[^a-z0-9]+", model_id.lower()))
        return (0 if tokens & small else 1, len(model_id))

    keep.sort(key=rank)
    return keep[:12]


def probe_hosted():
    """Try each provider's named candidates; if all fail, discover what the key
    can actually reach and try those. EVERY attempt is recorded, because the
    absence of a commercial model is itself a reportable finding."""
    print("\n" + "=" * 70); print("API TIER PROBE"); print("=" * 70)
    for name in list(HOSTED):
        info = HOSTED[name]
        try:
            client = make_client(name, info["key"])
        except Exception as e:
            why = classify_hosted_error(str(e))
            PROBE_LOG.append({"provider": name, "kind": info["kind"],
                              "model": None, "result": f"client init failed: {why}"})
            print(f"  {name:11s} client init failed: {why}")
            del HOSTED[name]; continue

        chosen = None
        attempts = [(m, p_in, p_out) for m, p_in, p_out in info["candidates"]]

        for phase in ("declared", "discovered"):
            if chosen:
                break
            if phase == "discovered":
                if not info.get("discover"):
                    break
                found = discover_models(name, client, info.get("discover_match", [""]))
                already = {m for m, _, _ in info["candidates"]}
                found = [f for f in found if f not in already]
                if not found:
                    continue
                print(f"  {name:11s} -- named candidates exhausted; discovered "
                      f"{len(found)} model(s) on this key, trying them --")
                # Price resolution for a discovered model:
                #   1. an explicit override, if the real published rate is known
                #   2. otherwise the cheapest declared input rate, with the
                #      output rate scaled up - flagged everywhere as an estimate
                base_in = min((p for _, p, _ in info["candidates"]), default=0.0)
                mult = CONFIG.get("estimated_output_multiplier", 4.0)
                attempts = []
                for f in found:
                    ov = CONFIG.get("price_overrides", {}).get(f)
                    if ov:
                        attempts.append((f, float(ov[0]), float(ov[1])))
                    else:
                        attempts.append((f, base_in, base_in * mult))
                        ESTIMATED_PRICES.add(f)

            for mid, p_in, p_out in attempts:
                try:
                    txt, _, _, _ = hosted_call_once(
                        name, client, mid, 'Reply with exactly one word: "security"')
                    if txt:
                        chosen = (mid, (p_in, p_out))
                        PROBE_LOG.append({"provider": name, "kind": info["kind"],
                                          "model": mid, "phase": phase,
                                          "result": "WORKS"})
                        print(f"  {name:11s} {mid:44s} WORKS ({phase})")
                        break
                    PROBE_LOG.append({"provider": name, "kind": info["kind"],
                                      "model": mid, "phase": phase,
                                      "result": "empty answer"})
                    print(f"  {name:11s} {mid:44s} empty answer")
                except Exception as e:
                    why = classify_hosted_error(str(e))
                    PROBE_LOG.append({"provider": name, "kind": info["kind"],
                                      "model": mid, "phase": phase, "result": why})
                    print(f"  {name:11s} {mid:44s} {why}")

        if chosen:
            info["client"], info["model"], info["price"] = client, chosen[0], chosen[1]
            HOSTED_PRICES[hosted_tag(name, chosen[0])] = chosen[1]
        else:
            del HOSTED[name]

    # Keep only models that actually WON their probe. The discovery loop adds
    # every candidate it merely priced to ESTIMATED_PRICES, so without this
    # the set - and the manifest field and verification detail derived from
    # it - lists models that never produced a single prediction.
    ESTIMATED_PRICES.intersection_update(
        {i.get("model") for i in HOSTED.values()})

    tiers = {i["kind"] for i in HOSTED.values()}
    print("\n  API models in play:")
    for name, info in HOSTED.items():
        est = " PRICE ESTIMATED" if info["model"] in ESTIMATED_PRICES else ""
        print(f"    {hosted_tag(name, info['model']):42s} [{info['kind']} / "
              f"{info['weights']}-weight]  "
              f"${info['price'][0]:.4f}/${info['price'][1]:.4f} per 1M tok{est}")

    live_est = [i["model"] for i in HOSTED.values() if i["model"] in ESTIMATED_PRICES]
    if live_est:
        print("\n  !! TOKEN PRICE IS AN ESTIMATE for: " + ", ".join(live_est))
        print("     These models were found by discovery, so their published rate")
        print("     is unknown to the harness. Accuracy results are unaffected,")
        print("     but the RQ3 cost axis for these models is indicative only.")
        print("     To fix: look up the published price and add it to")
        print("     CONFIG['price_overrides'], e.g.")
        for m in live_est:
            print(f"         \"{m}\": (0.10, 0.40),")
        print("     then re-run. NOTE: cost_usd is computed and FROZEN into each")
        print("     row at generation time (row_cost_usd), and scoring only sums")
        print("     the stored column - so a new override does NOT re-price rows")
        print("     already in the store. Either delete those rows so they are")
        print("     regenerated, or re-price them from the stored token counts")
        print("     (prompt_tokens / completion_tokens are kept for exactly this).")
    if "commercial" not in tiers:
        print("\n  !! NO COMMERCIAL (closed-weight) MODEL IS AVAILABLE.")
        print("     Both the named candidates AND live model discovery failed,")
        print("     so this is an account/quota limit, not a stale model name.")
        print("     Every attempt is written to commercial_availability.json.")
        print("     The thesis must reframe the tiers as 'local open-weight vs")
        print("     hosted open-weight vs fine-tuned' and cite that file in")
        print("     threats to validity. Do NOT label an open-weight model")
        print("     served over an API as 'commercial'.")
    print("=" * 70)

    with open(os.path.join(OUT, "commercial_availability.json"), "w") as f:
        json.dump({"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                   "commercial_model_obtained": "commercial" in tiers,
                   "discovery_attempted": True,
                   "probe_log": PROBE_LOG}, f, indent=2)


def hosted_call(provider, prompt, retries=4):
    info = HOSTED[provider]
    delay = 5.0
    for attempt in range(retries):
        try:
            return hosted_call_once(provider, info["client"], info["model"], prompt)
        except Exception as e:
            kind = classify_hosted_error(str(e))
            if kind == "rate limited":
                log.warning("%s rate limited; waiting %.0fs (%d/%d).",
                            provider, delay, attempt + 1, retries)
                time.sleep(delay); delay *= 2
            else:
                raise
    raise RuntimeError("rate limited after retries")


def run_hosted_provider(provider, store, rows):
    info = HOSTED[provider]
    tag, tier = hosted_tag(provider, info["model"]), info["kind"]
    interval = 60.0 / max(1, info.get("rpm", 10))
    todo = build_todo(tag, conditions_for(info["model"], is_hosted=True),
                      done_keys(store))
    # interleave across groups so a truncated run still covers every cell
    groups = {}
    for item in todo:
        groups.setdefault((item[1], item[2], item[0]["shot_k"]), []).append(item)
    todo, keys = [], list(groups)
    while any(groups[k] for k in keys):
        for k in keys:
            if groups[k]:
                todo.append(groups[k].pop(0))
    cap = int(info.get("daily_cap", 900))
    if len(todo) > cap:
        log.warning("[%s] %d requests queued but the free daily cap is %d. "
                    "Requests are interleaved across groups, so the truncated "
                    "run still covers every (task, dataset) cell evenly.",
                    tag, len(todo), cap)
    todo = todo[:cap]
    if not todo:
        log.info("[%s] nothing left within today's cap.", tag)
        return store

    log.info("[%s] %d requests (~%.0f min at %d RPM) [%s / %s-weight].",
             tag, len(todo), len(todo) * interval / 60,
             info.get("rpm", 10), tier, info["weights"])
    since, consecutive_fail = 0, 0
    for i, (cond, task, ds, exs, r) in enumerate(todo, 1):
        t_call = time.time()
        try:
            raw, n_in, n_out, dt = hosted_call(
                provider, build_prompt(task, r.text, cond["prompt_id"], exs))
            y_pred, ok = parse_label(task, raw)
            rows.append(make_row(tag, tier, cond, task, ds, r, y_pred, ok,
                                 n_in, n_out, dt, raw, "api_tokens",
                                 price=info.get("price", (0.0, 0.0))))
            consecutive_fail = 0
        except Exception as e:
            FAILURES.append({"model_tag": tag, "task": task, "dataset": ds,
                             "id": r.id, "prompt_id": cond["prompt_id"],
                             "shot_k": cond["shot_k"], "error": str(e)[:200]})
            consecutive_fail += 1
            if "no free quota" in classify_hosted_error(str(e)):
                log.warning("[%s] free quota exhausted - stopping provider.", tag); break
            if consecutive_fail >= 10:
                log.warning("[%s] ten consecutive failures - stopping.", tag); break
        since += 1
        if since >= CONFIG["flush_every"]:
            store = save_store(rows, store); since = 0
        if i % 50 == 0:
            log.info("[%s] %d/%d", tag, i, len(todo))
        wait = interval - (time.time() - t_call)
        if wait > 0:
            time.sleep(wait)
    return save_store(rows, store)


# =============================================================================
# statistics + summary
# =============================================================================
def bootstrap_ci(yt, yp, labelset, n_boot):
    yt, yp = np.asarray(yt), np.asarray(yp)
    if len(yt) < 5:
        return float("nan"), float("nan")
    rng = np.random.default_rng(SEED)
    stats = [f1_score(yt[i], yp[i], labels=labelset, average="macro", zero_division=0)
             for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n_boot))]
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)


def quarantine(store):
    """A model that cannot produce a parseable answer has not scored 0.000 - it
    has not been measured. Separating the two prevents a failed load from
    appearing in the results table as a legitimate result."""
    keys = ["model_tag", "task", "dataset", "prompt_id", "shot_k"]
    rates = store.groupby(keys).parse_ok.mean().reset_index(name="parse_rate")
    bad = rates[rates.parse_rate < CONFIG["min_parse_rate"]]
    if len(bad):
        bad = bad.assign(reason=f"parse_rate < {CONFIG['min_parse_rate']}")
        bad.to_csv(os.path.join(OUT, "stage3b_quarantine.csv"), index=False)
        log.warning("QUARANTINED %d model/condition cells (parse rate too low).", len(bad))
        merged = store.merge(bad[keys], on=keys, how="left", indicator=True)
        store = store[merged["_merge"].values == "left_only"]
    return store, bad


def summarise(store):
    keys = ["model_tag", "model_type", "split", "prompt_id", "shot_k", "task", "dataset"]
    recs = []
    for kv, g in store.groupby(keys):
        rec = dict(zip(keys, kv))
        labelset = LABELSETS[rec["task"]]
        # unparseable rows are excluded from the metric and reported separately
        ok = g[g.parse_ok]
        yt, yp = ok.y_true.astype(str).to_numpy(), ok.y_pred.astype(str).to_numpy()
        lo, hi = bootstrap_ci(yt, yp, labelset, CONFIG["n_boot"])
        tin, tout = int(g.prompt_tokens.sum()), int(g.completion_tokens.sum())
        # Summed from the stored per-row cost, so the aggregate can never drift
        # from the per-prediction figures in the appendix table.
        cost = float(g.cost_usd.sum())
        basis = g.cost_basis.iloc[0]
        ncorr = int((yt == yp).sum())
        rec.update({
            "n": len(g), "n_scored": len(ok),
            "macro_f1": round(f1_score(yt, yp, labels=labelset, average="macro", zero_division=0), 4),
            "weighted_f1": round(f1_score(yt, yp, labels=labelset, average="weighted", zero_division=0), 4),
            "ci_lo": lo, "ci_hi": hi,
            "accuracy": round(accuracy_score(yt, yp), 4),
            "macro_prec": round(precision_score(yt, yp, labels=labelset, average="macro", zero_division=0), 4),
            "macro_rec": round(recall_score(yt, yp, labels=labelset, average="macro", zero_division=0), 4),
            "parse_rate": round(g.parse_ok.mean(), 4),
            "latency_s_mean": round(float(g.latency_s.mean()), 4),
            "latency_s_p95": round(float(g.latency_s.quantile(0.95)), 4),
            "tok_in": tin, "tok_out": tout,
            "cost_basis": basis,
            "cost_usd": round(cost, 6),
            "cost_per_1k": round(cost / len(g) * 1000, 5),
            "cost_per_correct": round(cost / ncorr, 6) if ncorr else float("nan"),
        })
        recs.append(rec)
    return pd.DataFrame(recs).sort_values(
        ["task", "dataset", "split", "shot_k", "prompt_id", "macro_f1"],
        ascending=[True, True, True, True, True, False])


def fewshot_effect(store):
    """Few-shot deltas, ACTUALLY paired on identical items.

    This used to pivot the summary table, whose k=0 cell is the FULL frame
    while every k>0 cell is the CORE subset - different item sets, different
    class priors, and a delta that stage3b-repair (R2) showed flips sign in
    several cells once the sets are matched. It now works from the store: for
    each model, k=0 is re-scored on exactly the ids every k answered, under
    the strict convention (an unparseable answer is wrong, not absent).
    stage3b-repair's s3b_fix_fewshot_paired.csv adds the McNemar tests on the
    same pairing and is the authoritative table."""
    recs = []
    d = store[store.prompt_id == "base"]
    for (m, task, ds), g in d.groupby(["model_tag", "task", "dataset"]):
        ks = {int(k): x for k, x in g.groupby("shot_k")}
        if 0 not in ks or len(ks) < 2:
            continue
        shots = sorted(k for k in ks if k > 0)
        common = set.intersection(*[set(ks[k].id) for k in [0] + shots])
        if len(common) < 30:
            continue
        ls = LABELSETS[task]

        def strict_f1(x):
            x = x[x.id.isin(common)].drop_duplicates("id")
            yp = x.y_pred.where(x.parse_ok, "__unparsed__")
            return f1_score(x.y_true.astype(str), yp.astype(str), labels=ls,
                            average="macro", zero_division=0)

        f0 = strict_f1(ks[0])
        rec = {"model_tag": m, "task": task, "dataset": ds,
               "n_paired": len(common), "k=0": round(float(f0), 4)}
        for k in shots:
            fk = strict_f1(ks[k])
            rec[f"k={k}"] = round(float(fk), 4)
            rec[f"delta_k={k}"] = round(float(fk - f0), 4)
        recs.append(rec)
    return pd.DataFrame(recs)


# =============================================================================
# VERIFICATION  (run before Stage 3 may be marked DONE)
# =============================================================================
def export_unified_view(store):
    """Emit the 12-column appendix schema as a PROJECTION of the canonical store.

    It is deliberately not the store itself: Stages 2/4/5 require task, fold,
    prompt_id, shot_k, parse_ok and model_type, none of which appear in the
    12-column view. Shipping both gives the appendix a tidy table without
    removing the columns the pipeline cannot run without."""
    view = store.rename(columns=VIEW_RENAME)[UNIFIED_VIEW_COLS].copy()
    out = os.path.join(OUT, "predictions_subtype_harness")
    view.to_csv(out + ".csv", index=False)
    try:
        view.to_parquet(out + ".parquet", index=False)
    except Exception as e:
        log.warning("parquet view skipped (%s)", e)
    return view


def verify_stage3(store, summary, quarantined):
    """Hard gate. Every check must pass before Stage 4 is run."""
    checks, fails = [], []

    def chk(name, ok, detail=""):
        checks.append((name, bool(ok), detail))
        if not ok:
            fails.append(name)

    # ---- 1. schema -------------------------------------------------------
    chk("canonical schema complete",
        list(store.columns) == STORE_COLS,
        f"{len(store.columns)} cols")
    chk("split takes only zero_shot / few_shot",
        set(store.split.unique()) <= {"zero_shot", "few_shot"},
        str(sorted(store.split.unique())))
    chk("shot_k > 0 implies split == few_shot",
        bool(((store.shot_k > 0) == (store.split == "few_shot")).all()))
    chk("eval_regime populated", store.eval_regime.notna().all())
    chk("every row maps to a fine-tuned comparator fold",
        (store.fold != "unmapped").all(),
        str(sorted(store.fold.unique())))

    # ---- 2. instrumentation (RQ3) ---------------------------------------
    chk("latency: no NaN", store.latency_s.notna().all())
    chk("latency: no zeros", bool((store.latency_s > 0).all()),
        f"min={store.latency_s.min():.6f}")
    chk("prompt_tokens: all > 0", bool((store.prompt_tokens > 0).all()),
        f"min={int(store.prompt_tokens.min())}")
    chk("completion_tokens: no NaN", store.completion_tokens.notna().all())
    chk("cost_usd: no NaN", store.cost_usd.notna().all())
    # Zero is not automatically an error: OpenRouter ":free" routes bill $0.00
    # and CONFIG documents that 0.0 is their true rate, not a placeholder. The
    # old blanket "> 0 for every row" check therefore failed the whole run for
    # exactly the rows the config declares correct. The gate now requires
    # cost > 0 for every GPU-second row, and for API rows unless the row's own
    # stored prices declare the route free.
    _gpu = store[store.cost_basis == "gpu_seconds"]
    _api = store[store.cost_basis == "api_tokens"]
    chk("cost_usd > 0 for every GPU-second row",
        bool((_gpu.cost_usd > 0).all()) if len(_gpu) else True,
        f"min={_gpu.cost_usd.min():.3e}" if len(_gpu) else "no local rows")
    _api_ok = ((_api.cost_usd > 0) |
               ((_api.price_in_per_mtok == 0) & (_api.price_out_per_mtok == 0)))
    chk("cost_usd > 0 for every priced API row ($0 allowed on free routes)",
        bool(_api_ok.all()) if len(_api) else True,
        (f"{int((~_api_ok).sum())} zero-cost row(s) on a non-free route"
         if len(_api) and not _api_ok.all() else ""))
    # Only a model actually IN THE STORE can make the run unreportable.
    # ESTIMATED_PRICES accumulates every discovered candidate the probe merely
    # PRICED, so testing the whole set failed the run over ten models that
    # never produced a prediction. Store tags are "<provider>-<short-id>", so
    # the raw ids must be normalised the way hosted_tag() builds tags before
    # the two sets can be compared at all - the old comparison of raw ids
    # against stripped tags could never match anything.
    _est_short = {str(e).split("/")[-1].replace(":free", "").lower()
                  for e in ESTIMATED_PRICES}
    est_live = sorted(t for t in store.model_tag.astype(str).unique()
                      if re.sub(r"^(groq|openrouter)-", "", t).lower() in _est_short)
    chk("no model in the store is priced by estimate", not est_live,
        ("ESTIMATE used for: " + ", ".join(est_live) +
         " -> accuracy is unaffected; RQ3 cost for these is indicative only. "
         "Add the real rate to CONFIG['price_overrides'] and re-run scoring.")
        if est_live else
        (f"({len(ESTIMATED_PRICES)} probe-only candidates carry estimated "
         f"prices; none is in the store)" if ESTIMATED_PRICES else ""))
    chk("cost_basis is one of the two declared bases",
        set(store.cost_basis.unique()) <= {"gpu_seconds", "api_tokens"},
        str(sorted(store.cost_basis.unique())))
    chk("timestamp present", store.timestamp.notna().all())
    chk("quantization recorded", store.quantization.notna().all(),
        str(sorted(store.quantization.unique())))

    # ---- 3. coverage (RQ1) ----------------------------------------------
    core_models = set(store[store.split == "zero_shot"].model_tag)
    fs_models = set(store[store.split == "few_shot"].model_tag)
    local = set(store[store.model_type == "open_local"].model_tag)
    chk("every LOCAL model has few-shot data", local <= fs_models,
        f"missing: {sorted(local - fs_models)}")
    chk("all declared few-shot k values present",
        set(CONFIG["fewshot_ks"]) <= set(store[store.shot_k > 0].shot_k.unique()),
        str(sorted(store[store.shot_k > 0].shot_k.unique())))
    chk("at least one API-tier model ran",
        bool(len(store[store.model_type.isin(["open_hosted", "commercial"])])),
        "")

    # ---- 4. the core subset really is identical across models -----------
    # LOCAL models have no quota constraint, so a local model missing any core
    # item is a real defect and fails the gate. API models are capped by their
    # providers' free daily quotas BY DESIGN (Gemini's cap is 180 requests/day
    # against a 600-request core plan), so full coverage is impossible for
    # them by construction, and the old check failed every run that included
    # an API tier at all. Their coverage is REPORTED below instead, and the
    # like-for-like intersection tables (Stage 4 tab10 / stage3b-repair R7)
    # are what keep cross-tier comparisons valid on the items actually shared.
    core_sizes, api_core_coverage = {}, []
    _local_tags = set(store[store.model_type == "open_local"].model_tag)
    for (task, ds), fr in CORE.items():
        ids = set(fr.id)
        per_model = store[(store.task == task) & (store.dataset == ds) &
                          (store.split == "zero_shot") & (store.prompt_id == "base")]
        covered = per_model.groupby("model_tag").id.apply(lambda x: ids <= set(x))
        core_sizes[f"{task}/{ds}"] = int(len(ids))
        loc = covered[covered.index.isin(_local_tags)]
        chk(f"core fully covered by every LOCAL model: {task}/{ds}",
            bool(loc.all()) if len(loc) else False,
            f"n={len(ids)}, local models failing: {list(loc[~loc].index)}")
        for tag, x in per_model[~per_model.model_tag.isin(_local_tags)].groupby("model_tag"):
            api_core_coverage.append(
                f"{tag:38s} {task}/{ds}: {len(set(x.id) & ids)}/{len(ids)}")

    # ---- 5. no leakage between exemplars and evaluation items -----------
    leak = 0
    for key, pool in POOL.items():
        pool_texts = {t for t, _ in pool}
        frame_texts = set(FRAMES[key].text)
        leak += len(pool_texts & frame_texts)
    chk("few-shot exemplars excluded from evaluation sets", leak == 0, f"{leak} overlaps")

    # ---- 6. row reconciliation ------------------------------------------
    expected = {}
    for tag, g in store.groupby("model_tag"):
        expected[tag] = len(g)
    dupes = int(store.duplicated(
        subset=["model_tag", "task", "dataset", "prompt_id", "shot_k", "id"]).sum())
    chk("no duplicate predictions", dupes == 0, f"{dupes} duplicates")

    # ---- report ----------------------------------------------------------
    print("\n" + "=" * 74)
    print("STAGE 3 VERIFICATION")
    print("=" * 74)
    for name, ok, detail in checks:
        print(f"  {'PASS' if ok else 'FAIL'}  {name:52s} {detail}")

    print("\n  Rows per model:")
    for tag, n in sorted(expected.items(), key=lambda x: -x[1]):
        mt = store[store.model_tag == tag].model_type.iloc[0]
        pr = store[store.model_tag == tag].parse_ok.mean()
        print(f"    {tag:38s} {n:6d} rows  [{mt:11s}]  parse={pr:.1%}")

    print(f"\n  Core subset sizes: {core_sizes}")
    if api_core_coverage:
        print("  API-tier core coverage (quota-capped by design; cross-tier")
        print("  comparisons use the like-for-like intersection tables):")
        for line in api_core_coverage:
            print(f"    {line}")
    print(f"  Permanent failures excluded from the store: {len(FAILURES)}")
    print(f"  Quarantined cells (parse rate < {CONFIG['min_parse_rate']}): "
          f"{len(quarantined)}")
    print(f"  Total cost of the whole benchmark: ${store.cost_usd.sum():.4f}")

    print("\n" + "=" * 74)
    if fails:
        print(f"  RESULT: {len(fails)} CHECK(S) FAILED - DO NOT PROCEED TO STAGE 4")
        for f in fails:
            print(f"     - {f}")
    else:
        print("  RESULT: ALL CHECKS PASSED - Stage 3 is DONE, proceed to Stage 4.")
    print("=" * 74)
    return len(fails) == 0


def per_class_table(store):
    """Per-category precision/recall/F1 at the zero-shot base condition. For an
    11-way task the macro-F1 alone hides which classes fail; this is what lets
    the thesis say 'the tail classes (portability, legal) drive the gap'."""
    from sklearn.metrics import classification_report
    rows = []
    prim = store[(store.prompt_id == "base") & (store.shot_k == 0) & store.parse_ok]
    for (tag, task), g in prim.groupby(["model_tag", "task"]):
        labels = LABELSETS[task]
        rep = classification_report(g.y_true.astype(str), g.y_pred.astype(str),
                                    labels=labels, output_dict=True, zero_division=0)
        for c in labels:
            rows.append({"model_tag": tag, "task": task, "category": c,
                         "precision": round(rep[c]["precision"], 4),
                         "recall": round(rep[c]["recall"], 4),
                         "f1": round(rep[c]["f1-score"], 4),
                         "support": int(rep[c]["support"])})
    return pd.DataFrame(rows)


def majority_baseline():
    rows = []
    for (task, ds), fr in FRAMES.items():
        maj = fr.y_true.mode().iloc[0]
        yp = [maj] * len(fr)
        rows.append({"task": task, "dataset": ds, "n": len(fr), "majority_class": maj,
                     "macro_f1": round(f1_score(fr.y_true, yp, labels=LABELSETS[task],
                                                average="macro", zero_division=0), 4),
                     "accuracy": round(accuracy_score(fr.y_true, yp), 4)})
    return pd.DataFrame(rows)


# =============================================================================
# run
# =============================================================================
if CONFIG["run_hosted"]:
    probe_hosted()
    CONFIG["run_hosted"] = bool(HOSTED)

# SEED BEFORE PREFLIGHT. Preflight downloads and loads every local model to
# prove it can generate - tens of minutes and several GB on a cold Kaggle
# session. On a resumed run most models have nothing left to do, so the store
# is seeded first and only models with outstanding work are preflighted. A
# fully-resumed session therefore reaches the summaries in minutes instead of
# re-downloading six models to discover it has no work for them.
seed_store_from_inputs()
_SEEDED = load_store()
_ALL_LOCAL = list(CONFIG["open_models"])
if len(_SEEDED):
    _done = done_keys(_SEEDED)
    _needs = [m for m in _ALL_LOCAL
              if build_todo(m.split("/")[-1].lower(), conditions_for(m), _done)]
    COMPLETE_MODELS = [m for m in _ALL_LOCAL if m not in _needs]
    if COMPLETE_MODELS:
        log.info("Resume: %d of %d local model(s) already complete (%s) - their "
                 "preflight download is skipped entirely.",
                 len(COMPLETE_MODELS), len(_ALL_LOCAL),
                 ", ".join(m.split("/")[-1] for m in COMPLETE_MODELS))
else:
    _needs, COMPLETE_MODELS = _ALL_LOCAL, []

if _needs:
    CONFIG["open_models"], DROPPED_MODELS = preflight_models(_needs)
else:
    CONFIG["open_models"], DROPPED_MODELS = [], []
    print("\n  Every local model is already complete in the seeded store; "
          "skipping preflight.\n")
if not CONFIG["open_models"] and not HOSTED and not len(_SEEDED):
    raise RuntimeError("No usable model at all. Check the preflight output above.")


def smoke_test():
    """Two-minute rehearsal against a scratch store, so throwaway predictions
    can never reach the real results."""
    global STORE_PATH, FRAMES, CORE
    real_path, real_frames, real_core = STORE_PATH, FRAMES, CORE
    STORE_PATH = os.path.join(OUT, "_smoke_scratch")
    # min_per_class=1: the real floor of 10 would inflate a "tiny sample" far
    # beyond smoke_n_per_group (to ~109 items on the 11-class task).
    _smoke = {k: stratified(v, CONFIG["smoke_n_per_group"], min_per_class=1)
              for k, v in real_frames.items()}
    FRAMES, CORE = _smoke, dict(_smoke)
    print("\n" + "=" * 70); print("SMOKE TEST (tiny sample, throwaway numbers)"); print("=" * 70)
    try:
        s, r = pd.DataFrame(columns=STORE_COLS), []
        if CONFIG["open_models"]:
            s = run_open_model(CONFIG["open_models"][0], s, r)
        if len(s) == 0:
            raise RuntimeError("SMOKE TEST FAILED: no predictions produced.")
        pr = s.parse_ok.mean()
        print(f"  predictions   : {len(s)}")
        print(f"  parse rate    : {pr:.0%}")
        print(f"  latency > 0   : {(s.latency_s > 0).all()}")
        print(f"  tokens  > 0   : {(s.prompt_tokens > 0).all()}")
        print(f"  splits seen   : {sorted(s.split.unique())}")
        if pr < CONFIG["min_parse_rate"]:
            raise RuntimeError(f"SMOKE TEST FAILED: parse rate {pr:.0%}. Sample answers: "
                               f"{[str(x)[:60] for x in s.raw_output.head(3)]}")
        print("  SMOKE TEST PASSED"); print("=" * 70)
    finally:
        for ext in (".parquet", ".csv"):
            if os.path.exists(STORE_PATH + ext):
                os.remove(STORE_PATH + ext)
        STORE_PATH, FRAMES, CORE = real_path, real_frames, real_core


def full_run():
    """Two passes, by importance.

    PHASE 1 gives every model the core subset: zero-shot plus every few-shot k,
    on identical items. That is the complete, like-for-like results table the
    paper is built on. PHASE 2 adds the local-only extras (full evaluation sets,
    prompt sub-study).

    Running phase 1 for ALL models before ANY model starts phase 2 costs one
    extra model load each (~40 s) and buys a guarantee: if the session is cut
    short at any point after phase 1, the headline table is complete for every
    model rather than finished for three models and empty for three."""
    # Already seeded above, before preflight.
    store, rows = load_store(), []

    for phase, label in ((PHASE_PRIMARY, "PHASE 1 - core subset, all models"),
                         (PHASE_EXTRA, "PHASE 2 - full sets & prompt sub-study")):
        print("\n" + "=" * 70); print(label); print("=" * 70)
        for mid in CONFIG["open_models"]:
            try:
                store = run_open_model(mid, store, rows, phase=phase)
            except Exception as e:
                log.error("Model %s failed in phase %d (%s) - continuing.",
                          mid, phase, e)
                DROPPED_MODELS.append(
                    {"model": mid, "reason": f"runtime failure (phase {phase}): "
                                             f"{str(e)[:110]}"})
                store = save_store(rows, store)

        if phase == PHASE_PRIMARY:
            # API tier belongs to phase 1: its whole free quota is spent on the
            # comparable core condition.
            for provider in list(HOSTED):
                try:
                    store = run_hosted_provider(provider, store, rows)
                except Exception as e:
                    log.error("Provider %s failed (%s).", provider, e)
                    store = save_store(rows, store)
            log.info("PHASE 1 COMPLETE - the core comparison table is now safe. "
                     "Everything after this point is a robustness extra.")
    return store


# The smoke test exists to catch a broken generation path before a long run.
# With no local model left to run there is no such path to check, and running
# it would load a model purely to throw the result away.
if CONFIG["mode"] in ("auto", "smoke") and CONFIG["open_models"]:
    smoke_test()

if CONFIG["mode"] in ("auto", "full"):
    store = full_run()
    # The FULL store is what persists. Quarantine is a scoring-time filter,
    # not a deletion: physically dropping the rows would mean the committed
    # store loses them and a later resume re-runs those API calls, and Stage 4
    # applies its own MIN_PARSE_RATE gate to whatever it reads, so nothing
    # downstream needs them removed from disk. Every summary, the appendix
    # view and the verification gate score the quarantine-filtered view.
    write_table(store, STORE_PATH)
    scored, quarantined = quarantine(store)

    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
    summary = summarise(scored)

    print("\n" + "=" * 70)
    print("ZERO-SHOT, base prompt")
    print("Local models cover their FULL evaluation frames; API models cover")
    print("what their free daily quota allowed (see the n column). Cross-tier")
    print("rankings belong to the like-for-like tables, not to this one.")
    print("=" * 70)
    zs = summary[(summary.split == "zero_shot") & (summary.prompt_id == "base")]
    print(zs.drop(columns=["split", "prompt_id", "shot_k"]).to_string(index=False))

    print("\nMajority-class baseline:")
    print(majority_baseline().to_string(index=False))

    print("\n" + "=" * 70)
    print("FEW-SHOT EFFECT  (k=0 re-scored on the paired ids; strict scoring)")
    print("stage3b-repair's s3b_fix_fewshot_paired.csv adds McNemar tests on")
    print("this same pairing and is the authoritative table for the write-up.")
    print("=" * 70)
    fs = fewshot_effect(scored)
    print(fs.to_string(index=False))

    view = export_unified_view(scored)
    print(f"\n  Appendix view written: predictions_subtype_harness.parquet/.csv "
          f"({len(view)} rows x {len(view.columns)} cols)")

    STAGE3_OK = verify_stage3(scored, summary, quarantined)
    summary.to_csv(os.path.join(OUT, "subtype_summary.csv"), index=False)
    fs.to_csv(os.path.join(OUT, "subtype_fewshot_effect.csv"), index=False)
    majority_baseline().to_csv(os.path.join(OUT, "subtype_baseline.csv"), index=False)
    per_class_table(scored).to_csv(os.path.join(OUT, "subtype_perclass.csv"), index=False)
    pd.DataFrame(FAILURES).to_csv(os.path.join(OUT, "stage3b_failures.csv"), index=False)

    import transformers
    with open(os.path.join(OUT, "stage3b_manifest.json"), "w") as f:
        json.dump({"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": SEED,
                   "device": DEVICE, "python": platform.python_version(),
                   "torch": torch.__version__, "transformers": transformers.__version__,
                   "used_4bit": USE_4BIT,
                   "models_run": CONFIG["open_models"],
                   "models_already_complete_in_seeded_store": COMPLETE_MODELS,
                   "models_dropped": DROPPED_MODELS,
                   "api_models": {n: {"model": i.get("model"), "tier": i["kind"],
                                      "weights": i["weights"],
                                      "price_in_per_mtok": i.get("price", (0, 0))[0],
                                      "price_out_per_mtok": i.get("price", (0, 0))[1],
                                      "price_is_estimated":
                                          i.get("model") in ESTIMATED_PRICES}
                                  for n, i in HOSTED.items()},
                   "models_with_estimated_prices": sorted(ESTIMATED_PRICES),
                   "n_permanent_failures": len(FAILURES),
                   "n_quarantined_cells": int(len(quarantined)),
                   "config": CONFIG}, f, indent=2, default=str)

    print("\n" + "=" * 70)
    print("FILES WRITTEN"); print("=" * 70)
    for fn in sorted(os.listdir(OUT)):
        if fn.startswith(("predictions_subtype", "subtype_", "stage3b_", "commercial_")):
            print(f"  {fn:34s} {os.path.getsize(os.path.join(OUT, fn))/1024:8.1f} KB")
    print("\n  Reminder: outputs survive only in a SAVED VERSION.")
    print("  Save Version -> Save & Run All (Commit) -> Run with GPU.")
    print("=" * 70)
